In [1]:

# First, let me check the current directory structure to see if there's any dataset available
import os
import glob

# Check for any data files
data_extensions = ['*.csv', '*.parquet', '*.hdf5', '*.h5', '*.json', '*.zarr']
found_files = []

for ext in data_extensions:
 found_files.extend(glob.glob(ext))

print("Files found in current directory:")
for f in found_files:
 print(f" {f}")

if not found_files:
 print(" No data files found with expected extensions")

# List all files in current directory
print("\nAll files in directory:")
for item in os.listdir('.'):
 if os.path.isfile(item):
 print(f" {item}")


Files found in current directory:
 No data files found with expected extensions

All files in directory:
 research-program-v3.pdf
 main.pdf


In [2]:

# The dataset is described as computationally generated. 
# According to the description, I need to generate the data according to the specified protocol.
# Let me create the data generation pipeline following the requirements.

import numpy as np
from scipy import stats
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Tuple
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("ANALYSIS PLAN")
print("=" * 80)
print("""
OBJECTIVE: 
Compute single-scale metrics for D_F(t; N) time series across function classes
to test if non-multiplicative functions (DH family) show significantly higher 
kurtosis and resonance scores than multiplicative functions (ζ, L(χ₄), f_rand).

STEP 1: Data Generation
--------
Generate D_F(t; N) for five function classes:
1. Riemann ζ (multiplicative, coefficients a_n = 1)
2. L(s, χ₄) (multiplicative, real character mod 5)
3. f_rand (random multiplicative function)
4. L_DH (Davenport-Heilbronn, non-multiplicative)
5. L_DH^(ε) (perturbed DH family, non-multiplicative)

Parameters:
- N ∈ {10^4, 10^5, 10^6} (truncation lengths)
- T range: sample t ∈ [100, 200] with appropriate resolution Δt = 2π/log(N)
- Use Kahan compensated summation for numerical stability
- Validate against high-precision reference for N ≤ 10^4

STEP 2: Metric Computation
-----------
For each time series, compute:
1. Maximum magnitude: M_F = max_t |D_F(t; N)|
2. Mean square: V_F = mean_t |D_F(t; N)|²
3. Kurtosis: K_F = E[|D_F|⁴] / (E[|D_F|²])²
4. Resonance Score: RS(F) = M_F / √V_F
5. Tail probabilities: P_F(λ) for λ ∈ {2, 3, 4, 5}

STEP 3: Statistical Comparison
---------------
- Compile summary table of metrics by function class and N
- Perform statistical tests (Mann-Whitney U test) to assess separation
- Create visualization of kurtosis vs. function class

STEP 4: Validation
----------
Verify that DH function shows:
- RS(L_DH) >> RS(ζ)
- K_DH >> 3 (heavy tails)
- Power-law peak growth with N

STATISTICAL METHODS:
- Mann-Whitney U test (non-parametric, robust to non-normality)
- Effect size calculation (Cohen's d)
- Significance threshold: α = 0.05

POTENTIAL LIMITATIONS:
- Computational precision at large N
- Limited height range T ∈ [100, 200] may miss some phenomena
- Statistical power limited by number of function instances
""")
print("=" * 80)


ANALYSIS PLAN

OBJECTIVE: 
Compute single-scale metrics for D_F(t; N) time series across function classes
to test if non-multiplicative functions (DH family) show significantly higher 
kurtosis and resonance scores than multiplicative functions (ζ, L(χ₄), f_rand).

STEP 1: Data Generation
--------
Generate D_F(t; N) for five function classes:
1. Riemann ζ (multiplicative, coefficients a_n = 1)
2. L(s, χ₄) (multiplicative, real character mod 5)
3. f_rand (random multiplicative function)
4. L_DH (Davenport-Heilbronn, non-multiplicative)
5. L_DH^(ε) (perturbed DH family, non-multiplicative)

Parameters:
- N ∈ {10^4, 10^5, 10^6} (truncation lengths)
- T range: sample t ∈ [100, 200] with appropriate resolution Δt = 2π/log(N)
- Use Kahan compensated summation for numerical stability
- Validate against high-precision reference for N ≤ 10^4

STEP 2: Metric Computation
-----------
For each time series, compute:
1. Maximum magnitude: M_F = max_t |D_F(t; N)|
2. Mean square: V_F = mean_t |D_F(t; N

In [3]:

# STEP 1: Define coefficient generation functions for each function class

def generate_zeta_coefficients(N):
 """Riemann zeta: a_n = 1 for all n"""
 return np.ones(N, dtype=np.float64)

def generate_chi4_coefficients(N):
 """L(s, χ₄) with χ₄ the real character mod 5
 χ₄(1) = 1, χ₄(2) = -1, χ₄(3) = -1, χ₄(4) = 1, χ₄(0) = 0
 Extended multiplicatively
 """
 a = np.zeros(N, dtype=np.float64)
 # Define character on primes and extend multiplicatively
 chi4_values = {0: 0, 1: 1, 2: -1, 3: -1, 4: 1}
 
 # For n=1
 if N >= 1:
 a[0] = 1.0 # n=1
 
 # Generate multiplicatively
 for n in range(1, N):
 n_val = n + 1 # n starts from 1
 if n_val % 5 == 0:
 a[n] = 0
 else:
 # Compute χ₄(n) based on n mod 5
 a[n] = chi4_values[n_val % 5]
 
 return a

def generate_random_multiplicative_coefficients(N, seed=42):
 """Random multiplicative function: a_p = ±1 uniform i.i.d., extended multiplicatively"""
 rng = np.random.RandomState(seed)
 a = np.zeros(N, dtype=np.float64)
 a[0] = 1.0 # a_1 = 1
 
 # Sieve of Eratosthenes to find primes
 is_prime = np.ones(N + 1, dtype=bool)
 is_prime[0:2] = False
 for i in range(2, int(np.sqrt(N + 1)) + 1):
 if is_prime[i]:
 is_prime[i*i:N+1:i] = False
 
 primes = np.where(is_prime[:N+1])[0]
 
 # Assign random ±1 to each prime
 prime_values = {}
 for p in primes:
 if p >= 2:
 prime_values[p] = rng.choice([-1.0, 1.0])
 
 # Extend multiplicatively using prime factorization
 for n in range(1, N):
 n_val = n + 1
 if n_val == 1:
 a[n] = 1.0
 else:
 # Factorize n_val and compute product
 temp_n = n_val
 coeff = 1.0
 for p in primes:
 if p > temp_n:
 break
 while temp_n % p == 0:
 coeff *= prime_values.get(p, 1.0)
 temp_n //= p
 a[n] = coeff
 
 return a

def generate_DH_coefficients(N, epsilon=0.0):
 """Davenport-Heilbronn function coefficients
 L_DH(s) = ((1-iκ)/2) L(s,χ) + ((1+iκ)/2) L(s,χ̄)
 where χ is the complex character mod 5 of order 4:
 χ(1) = 1, χ(2) = i, χ(3) = -i, χ(4) = -1, χ(0) = 0
 and κ = (√5 - 1) / (2√(5(√5 - 1)))
 
 For perturbations, κ' = κ + ε
 """
 sqrt5 = np.sqrt(5.0)
 kappa = (sqrt5 - 1.0) / (2.0 * np.sqrt(5.0 * (sqrt5 - 1.0)))
 kappa_prime = kappa + epsilon
 
 # Complex character χ mod 5
 chi_values = {0: 0, 1: 1, 2: 1j, 3: -1j, 4: -1}
 
 a = np.zeros(N, dtype=np.complex128)
 
 for n in range(N):
 n_val = n + 1
 if n_val % 5 == 0:
 a[n] = 0.0 + 0.0j
 else:
 chi_n = chi_values[n_val % 5]
 chi_bar_n = np.conj(chi_n)
 a[n] = ((1 - 1j*kappa_prime) / 2.0) * chi_n + ((1 + 1j*kappa_prime) / 2.0) * chi_bar_n
 
 return a

# Test coefficient generation
print("Testing coefficient generation...")
print(f"ζ coefficients (n=1 to 10): {generate_zeta_coefficients(10)}")
print(f"χ₄ coefficients (n=1 to 10): {generate_chi4_coefficients(10)}")
print(f"f_rand coefficients (n=1 to 10): {generate_random_multiplicative_coefficients(10)}")
print(f"L_DH coefficients (n=1 to 10):\n{generate_DH_coefficients(10)}")
print("\nCoefficients generated successfully!")


Testing coefficient generation...
ζ coefficients (n=1 to 10): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
χ₄ coefficients (n=1 to 10): [ 1. -1. -1. 1. 0. 1. -1. -1. 1. 0.]
f_rand coefficients (n=1 to 10): [ 1. -1. 1. 1. -1. -1. -1. -1. 1. 1.]
L_DH coefficients (n=1 to 10):
[ 1. +0.j 0.24860289+0.j -0.24860289+0.j -1. +0.j
 0. +0.j 1. +0.j 0.24860289+0.j -0.24860289+0.j
 -1. +0.j 0. +0.j]

Coefficients generated successfully!


In [4]:

# STEP 2: Implement Kahan compensated summation for numerical stability

def kahan_sum(values):
 """
 Kahan compensated summation algorithm for improved numerical precision.
 Returns sum of complex array with reduced floating-point error.
 """
 if len(values) == 0:
 return 0.0 + 0.0j if np.iscomplexobj(values) else 0.0
 
 # For complex values, sum real and imaginary parts separately
 if np.iscomplexobj(values):
 real_sum = kahan_sum(values.real)
 imag_sum = kahan_sum(values.imag)
 return real_sum + 1j * imag_sum
 
 # Kahan algorithm for real values
 total = 0.0
 c = 0.0 # Compensation for lost low-order bits
 
 for val in values:
 y = val - c
 t = total + y
 c = (t - total) - y
 total = t
 
 return total

def compute_D_F(t_values, a_n, N):
 """
 Compute D_F(t; N) = Σ_{n≤N} a_n / n^(1/2 + it)
 
 Using Kahan compensated summation for numerical stability.
 
 Parameters:
 -----------
 t_values : array of time points
 a_n : coefficients for the function
 N : truncation length
 
 Returns:
 --------
 D_F : complex array of values at each t
 """
 n_values = np.arange(1, N + 1)
 n_sqrt = np.sqrt(n_values)
 
 D_F = np.zeros(len(t_values), dtype=np.complex128)
 
 for idx, t in enumerate(t_values):
 # Compute n^(-it) = exp(-it * log(n))
 phases = np.exp(-1j * t * np.log(n_values))
 # Compute a_n / n^(1/2 + it)
 terms = (a_n / n_sqrt) * phases
 # Use Kahan summation
 D_F[idx] = kahan_sum(terms)
 
 return D_F

# Test the computation with a small example
print("Testing D_F computation...")
t_test = np.array([100.0, 101.0])
a_test = generate_zeta_coefficients(100)
D_test = compute_D_F(t_test, a_test, 100)
print(f"D_ζ(t=100, N=100) = {D_test[0]:.6f}")
print(f"D_ζ(t=101, N=100) = {D_test[1]:.6f}")
print(f"|D_ζ(t=100, N=100)| = {np.abs(D_test[0]):.6f}")
print("\nD_F computation implemented successfully!")


Testing D_F computation...
D_ζ(t=100, N=100) = 2.767099-0.093697j
D_ζ(t=101, N=100) = 0.241208-0.905264j
|D_ζ(t=100, N=100)| = 2.768685

D_F computation implemented successfully!


In [5]:

# STEP 3: Generate time series data for all function classes
# Using smaller N values initially due to computational constraints

print("Generating time series data...")
print("=" * 80)

# Define parameters
N_values = [10**4, 10**5] # Starting with smaller N for feasibility
T_min, T_max = 100, 200

# We'll use different ranges for different N to capture the known DH zero near t≈85.7
# For the full analysis, we should include t ≈ 85.7, but starting with [100, 200]
# to match the specified range

results = []

# For each N value
for N in N_values:
 # Determine sampling resolution: Δt = 2π / log(N)
 delta_t = 2 * np.pi / np.log(N)
 print(f"\nN = {N:,}")
 print(f" Δt = {delta_t:.4f}")
 
 # Generate t grid
 t_values = np.arange(T_min, T_max, delta_t)
 n_samples = len(t_values)
 print(f" Number of samples: {n_samples}")
 
 # Function classes to compute
 function_classes = {
 'zeta': ('multiplicative', generate_zeta_coefficients(N)),
 'chi4': ('multiplicative', generate_chi4_coefficients(N)),
 'f_rand': ('multiplicative', generate_random_multiplicative_coefficients(N, seed=42)),
 'L_DH': ('non-multiplicative', generate_DH_coefficients(N, epsilon=0.0)),
 'L_DH_pert1': ('non-multiplicative', generate_DH_coefficients(N, epsilon=0.01)),
 }
 
 for func_name, (func_type, coeffs) in function_classes.items():
 print(f" Computing {func_name}...", end=' ')
 
 # Compute D_F(t; N)
 D_F = compute_D_F(t_values, coeffs, N)
 
 # Store results
 results.append({
 'N': N,
 'function': func_name,
 'type': func_type,
 't_values': t_values.copy(),
 'D_F': D_F.copy(),
 'mag_D_F': np.abs(D_F)
 })
 
 print(f"Done. |D_F| range: [{np.min(np.abs(D_F)):.4f}, {np.max(np.abs(D_F)):.4f}]")

print("\n" + "=" * 80)
print(f"Data generation complete! Total time series: {len(results)}")


Generating time series data...

N = 10,000
 Δt = 0.6822
 Number of samples: 147


 Computing zeta... 

Done. |D_F| range: [0.0982, 5.7642]
 Computing chi4... 

Done. |D_F| range: [0.0009, 8.4216]
 Computing f_rand... 

Done. |D_F| range: [0.2071, 15.8813]
 Computing L_DH... 

...

Done. |D_F| range: [1.1785, 7.9068]
 Computing chi4... 

Done. |D_F| range: [0.0064, 9.0704]
 Computing f_rand... 

Done. |D_F| range: [0.1166, 21.5047]
 Computing L_DH... 

Done. |D_F| range: [0.0065, 4.2200]
 Computing L_DH_pert1... 

Done. |D_F| range: [0.0058, 4.2303]

Data generation complete! Total time series: 10


In [6]:

# STEP 4: Compute single-scale metrics for each time series

def compute_metrics(D_F):
 """
 Compute single-scale metrics for a D_F time series.
 
 Returns:
 --------
 dict with keys:
 M_F: maximum magnitude
 V_F: mean square
 K_F: kurtosis
 RS: resonance score
 P_2, P_3, P_4, P_5: tail probabilities
 """
 mag = np.abs(D_F)
 mag2 = mag**2
 mag4 = mag**4
 
 # 1. Maximum magnitude
 M_F = np.max(mag)
 
 # 2. Mean square
 V_F = np.mean(mag2)
 
 # 3. Kurtosis: E[|D_F|⁴] / (E[|D_F|²])²
 E_mag4 = np.mean(mag4)
 K_F = E_mag4 / (V_F**2)
 
 # 4. Resonance Score: M_F / √V_F
 RS = M_F / np.sqrt(V_F)
 
 # 5. Tail probabilities: P_F(λ) = fraction of t where |D_F(t)| > λ·√V_F
 sqrt_V_F = np.sqrt(V_F)
 tail_probs = {}
 for lam in [2, 3, 4, 5]:
 threshold = lam * sqrt_V_F
 tail_probs[f'P_{lam}'] = np.mean(mag > threshold)
 
 return {
 'M_F': M_F,
 'V_F': V_F,
 'K_F': K_F,
 'RS': RS,
 **tail_probs
 }

# Compute metrics for all time series
print("Computing metrics for all time series...")
print("=" * 80)

metrics_data = []

for result in results:
 metrics = compute_metrics(result['D_F'])
 
 metrics_data.append({
 'N': result['N'],
 'function': result['function'],
 'type': result['type'],
 **metrics
 })
 
 print(f"N={result['N']:>6,}, {result['function']:>12s}: "
 f"M_F={metrics['M_F']:7.3f}, V_F={metrics['V_F']:6.3f}, "
 f"K_F={metrics['K_F']:6.3f}, RS={metrics['RS']:6.3f}")

print("=" * 80)

# Create DataFrame for easier analysis
df_metrics = pd.DataFrame(metrics_data)
print("\nMetrics summary:")
print(df_metrics.to_string(index=False))


Computing metrics for all time series...
N=10,000, zeta: M_F= 5.764, V_F= 4.156, K_F= 3.282, RS= 2.827
N=10,000, chi4: M_F= 8.422, V_F= 5.316, K_F= 5.011, RS= 3.653
N=10,000, f_rand: M_F= 15.881, V_F= 9.975, K_F= 8.254, RS= 5.028
N=10,000, L_DH: M_F= 4.421, V_F= 3.259, K_F= 2.537, RS= 2.449
N=10,000, L_DH_pert1: M_F= 4.432, V_F= 3.275, K_F= 2.537, RS= 2.449
N=100,000, zeta: M_F= 7.907, V_F=13.923, K_F= 1.799, RS= 2.119
N=100,000, chi4: M_F= 9.070, V_F= 5.309, K_F= 5.072, RS= 3.937
N=100,000, f_rand: M_F= 21.505, V_F=13.498, K_F=13.178, RS= 5.853
N=100,000, L_DH: M_F= 4.220, V_F= 3.244, K_F= 2.553, RS= 2.343
N=100,000, L_DH_pert1: M_F= 4.230, V_F= 3.260, K_F= 2.553, RS= 2.343

Metrics summary:
 N function type M_F V_F K_F RS P_2 P_3 P_4 P_5
 10000 zeta multiplicative 5.764190 4.156302 3.282041 2.827383 0.054422 0.000000 0.000000 0.000000
 10000 chi4 multiplicative 8.421562 5.316003 5.010592 3.652582 0.068027 0.027211 0.000000 0.000000
 10000 f_rand multiplicative 15.881279 9.974837 8.25

In [7]:

# STEP 5: Statistical analysis - compare multiplicative vs non-multiplicative

print("STATISTICAL ANALYSIS: Multiplicative vs Non-Multiplicative Functions")
print("=" * 80)

# Separate by type
mult_df = df_metrics[df_metrics['type'] == 'multiplicative']
non_mult_df = df_metrics[df_metrics['type'] == 'non-multiplicative']

print(f"\nMultiplicative functions (n={len(mult_df)}):")
print(mult_df[['function', 'N', 'K_F', 'RS']].to_string(index=False))

print(f"\nNon-multiplicative functions (n={len(non_mult_df)}):")
print(non_mult_df[['function', 'N', 'K_F', 'RS']].to_string(index=False))

# Summary statistics by type
print("\n" + "=" * 80)
print("SUMMARY STATISTICS BY TYPE")
print("=" * 80)

for metric in ['K_F', 'RS', 'M_F', 'V_F']:
 print(f"\n{metric}:")
 print(f" Multiplicative: mean={mult_df[metric].mean():.4f}, "
 f"std={mult_df[metric].std():.4f}, "
 f"median={mult_df[metric].median():.4f}")
 print(f" Non-multiplicative: mean={non_mult_df[metric].mean():.4f}, "
 f"std={non_mult_df[metric].std():.4f}, "
 f"median={non_mult_df[metric].median():.4f}")
 
 # Mann-Whitney U test (non-parametric test for difference in distributions)
 from scipy.stats import mannwhitneyu
 stat, p_value = mannwhitneyu(mult_df[metric], non_mult_df[metric], alternative='two-sided')
 print(f" Mann-Whitney U test: U={stat:.2f}, p-value={p_value:.4f}")
 
 # Effect size (Cohen's d approximation)
 mean_diff = mult_df[metric].mean() - non_mult_df[metric].mean()
 pooled_std = np.sqrt((mult_df[metric].std()**2 + non_mult_df[metric].std()**2) / 2)
 cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0
 print(f" Effect size (Cohen's d): {cohens_d:.4f}")
 
 # Interpretation
 if p_value < 0.05:
 print(f" *** SIGNIFICANT difference (p < 0.05) ***")
 else:
 print(f" No significant difference (p >= 0.05)")


STATISTICAL ANALYSIS: Multiplicative vs Non-Multiplicative Functions

Multiplicative functions (n=6):
function N K_F RS
 zeta 10000 3.282041 2.827383
 chi4 10000 5.010592 3.652582
 f_rand 10000 8.253542 5.028432
 zeta 100000 1.798850 2.118990
 chi4 100000 5.071549 3.936604
 f_rand 100000 13.178411 5.853304

Non-multiplicative functions (n=4):
 function N K_F RS
 L_DH 10000 2.536793 2.449176
L_DH_pert1 10000 2.537018 2.449169
 L_DH 100000 2.553063 2.342941
L_DH_pert1 100000 2.553258 2.342962

SUMMARY STATISTICS BY TYPE

K_F:
 Multiplicative: mean=6.0992, std=4.0848, median=5.0411
 Non-multiplicative: mean=2.5450, std=0.0094, median=2.5450
 Mann-Whitney U test: U=20.00, p-value=0.1143
 Effect size (Cohen's d): 1.2305
 No significant difference (p >= 0.05)

RS:
 Multiplicative: mean=3.9029, std=1.3765, median=3.7946
 Non-multiplicative: mean=2.3961, std=0.0613, median=2.3961
 Mann-Whitney U test: U=20.00, p-value=0.1143
 Effect size (Cohen's d): 1.5466
 No significant difference (p >= 0.0

In [8]:

# OBSERVATION: The results are NOT showing the expected pattern!
# The hypothesis predicted DH would have HIGHER kurtosis and RS than multiplicative functions,
# but we're seeing the OPPOSITE pattern - especially for f_rand which has very high values.
# 
# Critical issue: We're sampling t ∈ [100, 200], but the known DH zero is at t ≈ 85.7
# We need to sample near the known zero location to see the resonance!

print("CRITICAL OBSERVATION")
print("=" * 80)
print("The current results show OPPOSITE of the expected pattern:")
print(" - f_rand (multiplicative) has HIGHEST kurtosis and RS")
print(" - L_DH (non-multiplicative) has LOWEST kurtosis and RS")
print()
print("ROOT CAUSE: The known DH zero is at t ≈ 85.7, but we sampled t ∈ [100, 200]!")
print("We are MISSING the resonance peak!")
print()
print("SOLUTION: Re-generate data with t range covering the known DH zeros:")
print(" - t ≈ 85.7 (σ = 0.8085)")
print(" - t ≈ 114.2 (σ = 0.6508)")
print(" - t ≈ 166.5 (σ = 0.5744)")
print(" - t ≈ 176.7 (σ = 0.7243)")
print()
print("Let's re-run with t ∈ [80, 190] to capture these peaks...")
print("=" * 80)


CRITICAL OBSERVATION
The current results show OPPOSITE of the expected pattern:
 - f_rand (multiplicative) has HIGHEST kurtosis and RS
 - L_DH (non-multiplicative) has LOWEST kurtosis and RS

ROOT CAUSE: The known DH zero is at t ≈ 85.7, but we sampled t ∈ [100, 200]!
We are MISSING the resonance peak!

SOLUTION: Re-generate data with t range covering the known DH zeros:
 - t ≈ 85.7 (σ = 0.8085)
 - t ≈ 114.2 (σ = 0.6508)
 - t ≈ 166.5 (σ = 0.5744)
 - t ≈ 176.7 (σ = 0.7243)

Let's re-run with t ∈ [80, 190] to capture these peaks...


In [9]:

# Re-generate data with corrected t range to capture DH zeros
print("RE-GENERATING DATA with corrected t range...")
print("=" * 80)

# New parameters
N_values = [10**4, 10**5]
T_min, T_max = 80, 190 # Cover known DH zeros at 85.7, 114.2, 166.5, 176.7

results_corrected = []

for N in N_values:
 delta_t = 2 * np.pi / np.log(N)
 print(f"\nN = {N:,}")
 print(f" Δt = {delta_t:.4f}")
 
 t_values = np.arange(T_min, T_max, delta_t)
 n_samples = len(t_values)
 print(f" Number of samples: {n_samples}")
 print(f" t range: [{t_values[0]:.2f}, {t_values[-1]:.2f}]")
 
 function_classes = {
 'zeta': ('multiplicative', generate_zeta_coefficients(N)),
 'chi4': ('multiplicative', generate_chi4_coefficients(N)),
 'f_rand': ('multiplicative', generate_random_multiplicative_coefficients(N, seed=42)),
 'L_DH': ('non-multiplicative', generate_DH_coefficients(N, epsilon=0.0)),
 'L_DH_pert1': ('non-multiplicative', generate_DH_coefficients(N, epsilon=0.01)),
 'L_DH_pert2': ('non-multiplicative', generate_DH_coefficients(N, epsilon=-0.01)),
 }
 
 for func_name, (func_type, coeffs) in function_classes.items():
 print(f" Computing {func_name}...", end=' ')
 
 D_F = compute_D_F(t_values, coeffs, N)
 
 results_corrected.append({
 'N': N,
 'function': func_name,
 'type': func_type,
 't_values': t_values.copy(),
 'D_F': D_F.copy(),
 'mag_D_F': np.abs(D_F)
 })
 
 mag = np.abs(D_F)
 # Find peak location
 peak_idx = np.argmax(mag)
 peak_t = t_values[peak_idx]
 peak_mag = mag[peak_idx]
 
 print(f"Done. Peak: |D_F|={peak_mag:.4f} at t={peak_t:.2f}, "
 f"range: [{np.min(mag):.4f}, {np.max(mag):.4f}]")

print("\n" + "=" * 80)
print(f"Corrected data generation complete! Total time series: {len(results_corrected)}")


RE-GENERATING DATA with corrected t range...

N = 10,000
 Δt = 0.6822
 Number of samples: 162
 t range: [80.00, 189.83]


 Computing zeta... 

Done. Peak: |D_F|=6.1337 at t=172.10, range: [0.2724, 6.1337]
 Computing chi4... 

Done. Peak: |D_F|=9.0645 at t=185.74, range: [0.0115, 9.0645]
 Computing f_rand... 

Done. Peak: |D_F|=15.5960 at t=149.58, range: [0.2406, 15.5960]
 Computing L_DH... 

...

Done. Peak: |D_F|=8.7712 at t=185.88, range: [0.0079, 8.7712]
 Computing f_rand... 

Done. Peak: |D_F|=20.8585 at t=114.38, range: [0.0717, 20.8585]
 Computing L_DH... 

Done. Peak: |D_F|=4.3255 at t=165.14, range: [0.0403, 4.3255]
 Computing L_DH_pert1... 

Done. Peak: |D_F|=4.3362 at t=165.14, range: [0.0385, 4.3362]
 Computing L_DH_pert2... 

Done. Peak: |D_F|=4.3149 at t=165.14, range: [0.0426, 4.3149]

Corrected data generation complete! Total time series: 12


In [10]:

# Compute metrics for corrected data
print("Computing metrics for corrected data...")
print("=" * 80)

metrics_corrected = []

for result in results_corrected:
 metrics = compute_metrics(result['D_F'])
 
 metrics_corrected.append({
 'N': result['N'],
 'function': result['function'],
 'type': result['type'],
 **metrics
 })
 
 print(f"N={result['N']:>6,}, {result['function']:>12s}: "
 f"K_F={metrics['K_F']:6.3f}, RS={metrics['RS']:6.3f}, "
 f"M_F={metrics['M_F']:7.3f}, V_F={metrics['V_F']:6.3f}")

print("=" * 80)

df_corrected = pd.DataFrame(metrics_corrected)
print("\nCorrected metrics summary:")
print(df_corrected[['N', 'function', 'type', 'M_F', 'V_F', 'K_F', 'RS']].to_string(index=False))


Computing metrics for corrected data...
N=10,000, zeta: K_F= 2.565, RS= 2.417, M_F= 6.134, V_F= 6.438
N=10,000, chi4: K_F= 5.169, RS= 3.913, M_F= 9.064, V_F= 5.367
N=10,000, f_rand: K_F= 9.443, RS= 4.883, M_F= 15.596, V_F=10.200
N=10,000, L_DH: K_F= 2.500, RS= 2.419, M_F= 4.308, V_F= 3.172
N=10,000, L_DH_pert1: K_F= 2.500, RS= 2.419, M_F= 4.319, V_F= 3.188
N=10,000, L_DH_pert2: K_F= 2.500, RS= 2.419, M_F= 4.298, V_F= 3.157
N=100,000, zeta: K_F= 1.800, RS= 2.094, M_F= 5.956, V_F= 8.091
N=100,000, chi4: K_F= 5.104, RS= 3.781, M_F= 8.771, V_F= 5.381
N=100,000, f_rand: K_F=11.379, RS= 5.802, M_F= 20.859, V_F=12.926
N=100,000, L_DH: K_F= 2.485, RS= 2.426, M_F= 4.325, V_F= 3.180
N=100,000, L_DH_pert1: K_F= 2.485, RS= 2.426, M_F= 4.336, V_F= 3.196
N=100,000, L_DH_pert2: K_F= 2.485, RS= 2.426, M_F= 4.315, V_F= 3.165

Corrected metrics summary:
 N function type M_F V_F K_F RS
 10000 zeta multiplicative 6.133744 6.438058 2.565482 2.417398
 10000 chi4 multiplicative 9.064453 5.367472 5.169324 3.9

In [11]:

# Statistical comparison with corrected data
print("STATISTICAL COMPARISON: Corrected Data")
print("=" * 80)

mult_df_corr = df_corrected[df_corrected['type'] == 'multiplicative']
non_mult_df_corr = df_corrected[df_corrected['type'] == 'non-multiplicative']

print(f"\nMultiplicative functions (n={len(mult_df_corr)}):")
for metric in ['K_F', 'RS']:
 print(f" {metric}: mean={mult_df_corr[metric].mean():.4f}, "
 f"std={mult_df_corr[metric].std():.4f}, "
 f"range=[{mult_df_corr[metric].min():.4f}, {mult_df_corr[metric].max():.4f}]")

print(f"\nNon-multiplicative functions (n={len(non_mult_df_corr)}):")
for metric in ['K_F', 'RS']:
 print(f" {metric}: mean={non_mult_df_corr[metric].mean():.4f}, "
 f"std={non_mult_df_corr[metric].std():.4f}, "
 f"range=[{non_mult_df_corr[metric].min():.4f}, {non_mult_df_corr[metric].max():.4f}]")

print("\n" + "=" * 80)
print("HYPOTHESIS TESTING")
print("=" * 80)

for metric in ['K_F', 'RS', 'M_F', 'V_F']:
 print(f"\n{metric}:")
 
 # Mann-Whitney U test
 stat, p_value = mannwhitneyu(mult_df_corr[metric], non_mult_df_corr[metric], 
 alternative='two-sided')
 
 # One-sided test: multiplicative > non-multiplicative (for K_F and RS)
 if metric in ['K_F', 'RS']:
 stat_greater, p_greater = mannwhitneyu(mult_df_corr[metric], non_mult_df_corr[metric], 
 alternative='greater')
 print(f" Mann-Whitney U (mult > non-mult): U={stat_greater:.2f}, p={p_greater:.6f}")
 
 # Effect size
 mean_diff = mult_df_corr[metric].mean() - non_mult_df_corr[metric].mean()
 pooled_std = np.sqrt((mult_df_corr[metric].std()**2 + non_mult_df_corr[metric].std()**2) / 2)
 cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0
 
 print(f" Mean difference: {mean_diff:+.4f}")
 print(f" Effect size (Cohen's d): {cohens_d:.4f}")
 
 if metric in ['K_F', 'RS']:
 if p_greater < 0.05:
 print(f" *** Multiplicative > Non-multiplicative (p = {p_greater:.6f}) ***")
 else:
 print(f" No significant difference (p = {p_greater:.6f})")

print("\n" + "=" * 80)
print("INTERPRETATION")
print("=" * 80)
print("""
RESULT: The hypothesis is REJECTED!

Expected: Non-multiplicative (DH) >> Multiplicative (ζ, χ₄, f_rand)
Observed: Multiplicative >> Non-multiplicative

Specifically:
 - Random multiplicative f_rand shows HIGHEST kurtosis and RS
 - DH family shows LOWEST kurtosis and RS
 - ζ and χ₄ show intermediate values

This is the OPPOSITE of what the hypothesis predicted!
""")


STATISTICAL COMPARISON: Corrected Data

Multiplicative functions (n=6):
 K_F: mean=5.9101, std=3.7858, range=[1.8004, 11.3787]
 RS: mean=3.8150, std=1.4152, range=[2.0938, 5.8017]

Non-multiplicative functions (n=6):
 K_F: mean=2.4926, std=0.0085, range=[2.4846, 2.5005]
 RS: mean=2.4224, std=0.0036, range=[2.4191, 2.4257]

HYPOTHESIS TESTING

K_F:
 Mann-Whitney U (mult > non-mult): U=30.00, p=0.032468
 Mean difference: +3.4175
 Effect size (Cohen's d): 1.2766
 *** Multiplicative > Non-multiplicative (p = 0.032468) ***

RS:
 Mann-Whitney U (mult > non-mult): U=24.00, p=0.196970
 Mean difference: +1.3926
 Effect size (Cohen's d): 1.3915
 No significant difference (p = 0.196970)

M_F:
 Mean difference: +6.7462
 Effect size (Cohen's d): 1.6075

V_F:
 Mean difference: +4.8912
 Effect size (Cohen's d): 2.2977

INTERPRETATION

RESULT: The hypothesis is REJECTED!

Expected: Non-multiplicative (DH) >> Multiplicative (ζ, χ₄, f_rand)
Observed: Multiplicative >> Non-multiplicative

Specifically:
 

In [12]:

# Let's investigate WHY this is happening
# Check if we're actually seeing the DH resonance peaks at the known zero locations

print("DIAGNOSTIC ANALYSIS: Are we capturing DH resonance peaks?")
print("=" * 80)

# Known DH zeros
dh_zeros = [
 (0.8085, 85.70),
 (0.6508, 114.16),
 (0.5744, 166.48),
 (0.7243, 176.70)
]

print("\nKnown DH zeros:")
for sigma, t in dh_zeros:
 print(f" σ = {sigma:.4f}, t = {t:.2f}")

print("\n" + "=" * 80)
print("Peak Analysis by Function")
print("=" * 80)

for result in results_corrected:
 if result['N'] == 10**5: # Focus on N=100,000
 mag = result['mag_D_F']
 t_vals = result['t_values']
 
 # Find top 5 peaks
 peak_indices = np.argsort(mag)[-5:][::-1]
 
 print(f"\n{result['function']} (N={result['N']:,}):")
 print(f" Top 5 peaks:")
 for i, idx in enumerate(peak_indices):
 print(f" {i+1}. t = {t_vals[idx]:.2f}, |D_F| = {mag[idx]:.4f}")
 
 # Check if near known DH zeros (only for DH functions)
 if 'L_DH' in result['function']:
 print(f" Distance to known DH zeros:")
 for sigma, t_zero in dh_zeros:
 # Find closest t in our grid
 dist = np.abs(t_vals - t_zero)
 min_idx = np.argmin(dist)
 closest_t = t_vals[min_idx]
 mag_at_zero = mag[min_idx]
 print(f" t = {t_zero:.2f}: closest grid point t = {closest_t:.2f}, "
 f"|D_F| = {mag_at_zero:.4f}, distance = {dist[min_idx]:.2f}")

print("\n" + "=" * 80)


DIAGNOSTIC ANALYSIS: Are we capturing DH resonance peaks?

Known DH zeros:
 σ = 0.8085, t = 85.70
 σ = 0.6508, t = 114.16
 σ = 0.5744, t = 166.48
 σ = 0.7243, t = 176.70

Peak Analysis by Function

zeta (N=100,000):
 Top 5 peaks:
 1. t = 109.47, |D_F| = 5.9556
 2. t = 172.23, |D_F| = 5.7694
 3. t = 91.46, |D_F| = 5.7039
 4. t = 110.02, |D_F| = 5.6349
 5. t = 82.18, |D_F| = 5.5973

chi4 (N=100,000):
 Top 5 peaks:
 1. t = 185.88, |D_F| = 8.7712
 2. t = 140.58, |D_F| = 8.0247
 3. t = 185.33, |D_F| = 7.6657
 4. t = 94.74, |D_F| = 6.9603
 5. t = 123.11, |D_F| = 6.2032

f_rand (N=100,000):
 Top 5 peaks:
 1. t = 114.38, |D_F| = 20.8585
 2. t = 149.31, |D_F| = 16.1359
 3. t = 177.14, |D_F| = 12.3301
 4. t = 85.46, |D_F| = 12.0419
 5. t = 95.28, |D_F| = 11.0787

L_DH (N=100,000):
 Top 5 peaks:
 1. t = 165.14, |D_F| = 4.3255
 2. t = 115.47, |D_F| = 4.3056
 3. t = 133.48, |D_F| = 4.0299
 4. t = 84.37, |D_F| = 3.9735
 5. t = 101.83, |D_F| = 3.8623
 Distance to known DH zeros:
 t = 85.70: closest g

In [13]:

# CRITICAL FINDING: At the known DH zero locations, |D_F| is SMALL (~0.3-0.6), not large!
# This is counter-intuitive. Let me check the theory...
#
# Wait - I think I misunderstood the signature!
# The off-line zero at σ = 0.8085 means the zero is at s = 0.8085 + i*85.7
# The resonance detector D_F(t; N) is evaluated at the CRITICAL LINE σ = 1/2
# So we're NOT at the zero location itself!
#
# The peak should grow as N^(σ - 1/2) = N^(0.8085 - 0.5) = N^0.3085
# Let's check if we see this scaling behavior!

print("SCALING ANALYSIS: Peak growth with N")
print("=" * 80)

# Group results by function
functions = df_corrected['function'].unique()

print("\nPeak magnitude M_F vs N:")
print("Function N=10^4 N=10^5 Ratio Expected (if power law)")
print("-" * 80)

for func in functions:
 data = df_corrected[df_corrected['function'] == func]
 if len(data) == 2:
 M_N1 = data[data['N'] == 10**4]['M_F'].values[0]
 M_N2 = data[data['N'] == 10**5]['M_F'].values[0]
 ratio = M_N2 / M_N1
 
 # For DH with σ ≈ 0.8085, expect ratio = (10^5/10^4)^(0.8085-0.5) = 10^0.3085 ≈ 2.03
 if 'L_DH' in func:
 expected_DH = 10**0.3085
 print(f"{func:12s} {M_N1:7.4f} {M_N2:7.4f} {ratio:5.3f} "
 f"{expected_DH:5.3f} (σ=0.8085)")
 else:
 # For functions on critical line, expect O(1) or logarithmic growth
 expected_log = np.sqrt(np.log(10**5) / np.log(10**4)) # ~ 1.07
 print(f"{func:12s} {M_N1:7.4f} {M_N2:7.4f} {ratio:5.3f} "
 f"{expected_log:5.3f} (log-growth)")

print("\n" + "=" * 80)
print("OBSERVATION:")
print(" - DH functions show ratio ≈ 1.00 (NO growth!)")
print(" - This should be ≈ 2.03 if there's power-law growth from off-line zero")
print(" - Instead, ζ, χ₄, f_rand show some growth")
print("\nPOSSIBLE EXPLANATIONS:")
print(" 1. The height range [80, 190] is not capturing the resonance properly")
print(" 2. The known zeros are not producing the expected signature in D_F(t; N)")
print(" 3. Our implementation of D_F is incorrect")
print(" 4. The theoretical prediction needs re-examination")
print("=" * 80)


SCALING ANALYSIS: Peak growth with N

Peak magnitude M_F vs N:
Function N=10^4 N=10^5 Ratio Expected (if power law)
--------------------------------------------------------------------------------
zeta 6.1337 5.9556 0.971 1.118 (log-growth)
chi4 9.0645 8.7712 0.968 1.118 (log-growth)
f_rand 15.5960 20.8585 1.337 1.118 (log-growth)
L_DH 4.3085 4.3255 1.004 2.035 (σ=0.8085)
L_DH_pert1 4.3189 4.3362 1.004 2.035 (σ=0.8085)
L_DH_pert2 4.2984 4.3149 1.004 2.035 (σ=0.8085)

OBSERVATION:
 - DH functions show ratio ≈ 1.00 (NO growth!)
 - This should be ≈ 2.03 if there's power-law growth from off-line zero
 - Instead, ζ, χ₄, f_rand show some growth

POSSIBLE EXPLANATIONS:
 1. The height range [80, 190] is not capturing the resonance properly
 2. The known zeros are not producing the expected signature in D_F(t; N)
 3. Our implementation of D_F is incorrect
 4. The theoretical prediction needs re-examination


In [14]:

# Let me re-read the research program document more carefully...
# The key is from page 10: "For L_DH at t* ≈ γ (off-line zero): 
# A peak at t = γ_0 with amplitude ~ N^(β_0 - 1/2), growing as N increases"
#
# So the peak should be AT the height t where the imaginary part of the zero is located.
# Let me check the magnitude specifically at t ≈ 85.7

print("DETAILED ANALYSIS: |D_F(t)| near known DH zero at t ≈ 85.7")
print("=" * 80)

# Focus on the first known zero at t ≈ 85.7
target_t = 85.7

for result in results_corrected:
 if 'L_DH' in result['function']:
 t_vals = result['t_values']
 mag = result['mag_D_F']
 
 # Find values near t ≈ 85.7
 mask = (t_vals >= 83) & (t_vals <= 88)
 t_near = t_vals[mask]
 mag_near = mag[mask]
 
 print(f"\n{result['function']} (N={result['N']:,}):")
 print(f" t values near 85.7:")
 for t, m in zip(t_near, mag_near):
 print(f" t = {t:.2f}, |D_F| = {m:.4f}")
 
 # Find peak in this region
 max_idx = np.argmax(mag_near)
 print(f" Local maximum: t = {t_near[max_idx]:.2f}, |D_F| = {mag_near[max_idx]:.4f}")

print("\n" + "=" * 80)
print("CRITICAL INSIGHT:")
print(" The magnitude at t ≈ 85.7 is relatively SMALL (< 1.0)")
print(" BUT this is expected for N ~ 10^5!")
print(" The peak should grow as N^(0.8085 - 0.5) = N^0.3085")
print(" For N = 10^5: peak ~ (10^5)^0.3085 ≈ 51.5")
print(" But we're only seeing ~0.6!")
print("\nThis suggests our data is NOT showing the expected DH resonance!")
print("=" * 80)


DETAILED ANALYSIS: |D_F(t)| near known DH zero at t ≈ 85.7

L_DH (N=10,000):
 t values near 85.7:
 t = 83.41, |D_F| = 1.5306
 t = 84.09, |D_F| = 4.0167
 t = 84.78, |D_F| = 2.8412
 t = 85.46, |D_F| = 0.5762
 t = 86.14, |D_F| = 0.9267
 t = 86.82, |D_F| = 2.2244
 t = 87.50, |D_F| = 0.6760
 Local maximum: t = 84.09, |D_F| = 4.0167

L_DH_pert1 (N=10,000):
 t values near 85.7:
 t = 83.41, |D_F| = 1.5343
 t = 84.09, |D_F| = 4.0268
 t = 84.78, |D_F| = 2.8480
 t = 85.46, |D_F| = 0.5777
 t = 86.14, |D_F| = 0.9281
 t = 86.82, |D_F| = 2.2300
 t = 87.50, |D_F| = 0.6778
 Local maximum: t = 84.09, |D_F| = 4.0268

L_DH_pert2 (N=10,000):
 t values near 85.7:
 t = 83.41, |D_F| = 1.5270
 t = 84.09, |D_F| = 4.0069
 t = 84.78, |D_F| = 2.8347
 t = 85.46, |D_F| = 0.5748
 t = 86.14, |D_F| = 0.9257
 t = 86.82, |D_F| = 2.2191
 t = 87.50, |D_F| = 0.6742
 Local maximum: t = 84.09, |D_F| = 4.0069

L_DH (N=100,000):
 t values near 85.7:
 t = 83.27, |D_F| = 0.8120
 t = 83.82, |D_F| = 3.3886
 t = 84.37, |D_F| = 3.973

In [15]:

# I need to reconsider the theory. Let me check if I'm computing D_F correctly.
# According to the document (page 3), the formula is:
#
# D_F(t; N) = Σ_{n≤N} a_n(F) / n^(1/2 + it)
#
# This can be rewritten as:
# D_F(t; N) = Σ_{n≤N} a_n(F) / (n^(1/2) * n^(it))
# = Σ_{n≤N} (a_n(F) / n^(1/2)) * n^(-it)
# = Σ_{n≤N} (a_n(F) / n^(1/2)) * exp(-it * log(n))
#
# This is what I implemented. Let me verify with a simple test.

print("VERIFICATION: Testing D_F computation")
print("=" * 80)

# Test with N=3, t=1.0, a_n = [1, 1, 1] (zeta)
N_test = 3
t_test = 1.0
a_test = np.ones(N_test)

# Manual computation
n_values = np.arange(1, N_test + 1)
manual_sum = 0.0 + 0.0j
for i, n in enumerate(n_values):
 term = a_test[i] / (n**(0.5 + 1.0j*t_test))
 manual_sum += term
 print(f"n={n}: a_n={a_test[i]}, n^(1/2+it)={n**(0.5+1.0j*t_test):.6f}, "
 f"term={term:.6f}")

print(f"\nManual sum: {manual_sum:.6f}")

# Using my function
D_computed = compute_D_F(np.array([t_test]), a_test, N_test)[0]
print(f"Computed D_F: {D_computed:.6f}")
print(f"Match: {np.allclose(manual_sum, D_computed)}")

print("\n" + "=" * 80)
print("Implementation is CORRECT!")
print("=" * 80)


VERIFICATION: Testing D_F computation
n=1: a_n=1.0, n^(1/2+it)=1.000000+0.000000j, term=1.000000+0.000000j
n=2: a_n=1.0, n^(1/2+it)=1.087868+0.903628j, term=0.543934-0.451814j
n=3: a_n=1.0, n^(1/2+it)=0.787793+1.542525j, term=0.262598-0.514175j

Manual sum: 1.806532-0.965989j
Computed D_F: 1.806532-0.965989j
Match: True

Implementation is CORRECT!


In [16]:

# The implementation is correct. The issue is that we're working with FINITE N.
# 
# Re-reading the validation criterion (page 4 of research-program-v3.pdf):
# "For L_DH at t* ≈ γ (off-line zero): Pers(t*; N1, N2) ~ (N2/N1)^(β0-1/2), growing with scale"
#
# This is about PERSISTENCE across scales, not absolute magnitude!
# The absolute peak magnitude may be modest, but it should GROW with N.
#
# Let me compute persistence and also try larger N values.
# But wait - we already saw that the ratio M_F(N=10^5) / M_F(N=10^4) ≈ 1.00 for DH,
# which is NOT growing!
#
# Let me think about this differently. Perhaps the issue is that:
# 1. We're sampling at discrete t values
# 2. The actual zero might not align perfectly with our grid
# 3. We need to look at the OVERALL distribution, not just peaks
#
# Actually, let me reconsider the ENTIRE hypothesis. The research says:
# "If the kurtosis of the DH function is not substantially larger than that of the zeta 
# function, the core premise of using these metrics to detect resonance is flawed."
#
# Our data shows K_DH ≈ 2.5, K_zeta ≈ 2.0, K_chi4 ≈ 5.1, K_f_rand ≈ 10.4
#
# So K_DH is NOT larger than K_zeta - in fact it's slightly larger.
# But K_f_rand >> K_DH!
#
# This suggests that the random multiplicative function has MORE heavy-tailed behavior
# than the DH function! This is unexpected but this IS what we observe.

print("SUMMARY OF FINDINGS")
print("=" * 80)
print("\nKurtosis (K_F) by function type:")
print(" ζ (Riemann zeta): K ≈ 2.0 - 2.6")
print(" χ₄ (real character): K ≈ 5.1 - 5.2")
print(" f_rand (random mult): K ≈ 9.4 - 11.4")
print(" L_DH (Davenport-Heilbronn): K ≈ 2.5")
print()
print("Resonance Score (RS) by function type:")
print(" ζ (Riemann zeta): RS ≈ 2.1 - 2.4")
print(" χ₄ (real character): RS ≈ 3.8 - 3.9")
print(" f_rand (random mult): RS ≈ 4.9 - 5.8")
print(" L_DH (Davenport-Heilbronn): RS ≈ 2.4")
print()
print("=" * 80)
print("KEY OBSERVATIONS:")
print("=" * 80)
print("1. The hypothesis predicted: K_DH >> K_multiplicative")
print(" Reality: K_DH < K_multiplicative (especially K_f_rand)")
print()
print("2. The hypothesis predicted: RS_DH >> RS_multiplicative")
print(" Reality: RS_DH < RS_multiplicative (especially RS_f_rand)")
print()
print("3. Random multiplicative function (f_rand) shows:")
print(" - HIGHEST kurtosis (K ≈ 10-11)")
print(" - HIGHEST resonance score (RS ≈ 5-6)")
print(" - HIGHEST tail probabilities")
print()
print("4. DH function shows:")
print(" - LOW kurtosis (K ≈ 2.5, similar to ζ)")
print(" - LOW resonance score (RS ≈ 2.4, similar to ζ)")
print(" - LOW tail probabilities")
print()
print("5. NO power-law peak growth observed for DH:")
print(" - Expected: M_F(10^5)/M_F(10^4) ≈ 2.03")
print(" - Observed: ratio ≈ 1.00")
print()
print("=" * 80)
print("CONCLUSION: The hypothesis is REJECTED")
print("=" * 80)


SUMMARY OF FINDINGS

Kurtosis (K_F) by function type:
 ζ (Riemann zeta): K ≈ 2.0 - 2.6
 χ₄ (real character): K ≈ 5.1 - 5.2
 f_rand (random mult): K ≈ 9.4 - 11.4
 L_DH (Davenport-Heilbronn): K ≈ 2.5

Resonance Score (RS) by function type:
 ζ (Riemann zeta): RS ≈ 2.1 - 2.4
 χ₄ (real character): RS ≈ 3.8 - 3.9
 f_rand (random mult): RS ≈ 4.9 - 5.8
 L_DH (Davenport-Heilbronn): RS ≈ 2.4

KEY OBSERVATIONS:
1. The hypothesis predicted: K_DH >> K_multiplicative
 Reality: K_DH < K_multiplicative (especially K_f_rand)

2. The hypothesis predicted: RS_DH >> RS_multiplicative
 Reality: RS_DH < RS_multiplicative (especially RS_f_rand)

3. Random multiplicative function (f_rand) shows:
 - HIGHEST kurtosis (K ≈ 10-11)
 - HIGHEST resonance score (RS ≈ 5-6)
 - HIGHEST tail probabilities

4. DH function shows:
 - LOW kurtosis (K ≈ 2.5, similar to ζ)
 - LOW resonance score (RS ≈ 2.4, similar to ζ)
 - LOW tail probabilities

5. NO power-law peak growth observed for DH:
 - Expected: M_F(10^5)/M_F(10^4) ≈ 2

In [17]:

# Create comprehensive summary table and save results

print("Creating comprehensive summary table...")
print("=" * 80)

# Create summary by function class (averaging over perturbations)
summary_data = []

for func in ['zeta', 'chi4', 'f_rand']:
 for N in [10**4, 10**5]:
 row = df_corrected[(df_corrected['function'] == func) & (df_corrected['N'] == N)].iloc[0]
 summary_data.append({
 'Function': func,
 'Type': 'Multiplicative',
 'N': N,
 'M_F': row['M_F'],
 'V_F': row['V_F'],
 'K_F': row['K_F'],
 'RS': row['RS'],
 'P_2': row['P_2'],
 'P_3': row['P_3'],
 'P_4': row['P_4'],
 'P_5': row['P_5']
 })

# For DH, average over perturbations
for N in [10**4, 10**5]:
 dh_rows = df_corrected[df_corrected['function'].str.contains('L_DH') & (df_corrected['N'] == N)]
 summary_data.append({
 'Function': 'L_DH (avg)',
 'Type': 'Non-Multiplicative',
 'N': N,
 'M_F': dh_rows['M_F'].mean(),
 'V_F': dh_rows['V_F'].mean(),
 'K_F': dh_rows['K_F'].mean(),
 'RS': dh_rows['RS'].mean(),
 'P_2': dh_rows['P_2'].mean(),
 'P_3': dh_rows['P_3'].mean(),
 'P_4': dh_rows['P_4'].mean(),
 'P_5': dh_rows['P_5'].mean()
 })

df_summary = pd.DataFrame(summary_data)

print("\nFINAL SUMMARY TABLE")
print("=" * 80)
print(df_summary.to_string(index=False))

# Save to CSV
df_summary.to_csv('resonance_metrics_summary.csv', index=False)
print("\nSaved summary to: resonance_metrics_summary.csv")

# Also save full metrics
df_corrected.to_csv('resonance_metrics_full.csv', index=False)
print("Saved full metrics to: resonance_metrics_full.csv")


Creating comprehensive summary table...

FINAL SUMMARY TABLE
 Function Type N M_F V_F K_F RS P_2 P_3 P_4 P_5
 zeta Multiplicative 10000 6.133744 6.438058 2.565482 2.417398 0.049383 0.000000 0.000000 0.00000
 zeta Multiplicative 100000 5.955569 8.090642 1.800447 2.093784 0.014851 0.000000 0.000000 0.00000
 chi4 Multiplicative 10000 9.064453 5.367472 5.169324 3.912521 0.067901 0.012346 0.000000 0.00000
 chi4 Multiplicative 100000 8.771245 5.381442 5.103539 3.781045 0.064356 0.019802 0.000000 0.00000
 f_rand Multiplicative 10000 15.596015 10.200063 9.443062 4.883287 0.049383 0.024691 0.012346 0.00000
 f_rand Multiplicative 100000 20.858507 12.925893 11.378748 5.801669 0.054455 0.024752 0.009901 0.00495
L_DH (avg) Non-Multiplicative 10000 4.308588 3.172105 2.500304 2.419146 0.055556 0.000000 0.000000 0.00000
L_DH (avg) Non-Multiplicative 100000 4.325549 3.180046 2.484853 2.425635 0.049505 0.000000 0.000000 0.00000

Saved summary to: resonance_metrics_summary.csv
Saved full metrics to: reso

In [18]:

# Create final visualization summarizing the key finding
import matplotlib.pyplot as plt

# Set up publication-quality figure
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 14

# Create figure with single column layout (vertically stacked subfigures)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10))

# Prepare data for plotting
funcs_ordered = ['zeta', 'chi4', 'f_rand', 'L_DH (avg)']
types = ['Multiplicative', 'Multiplicative', 'Multiplicative', 'Non-Multiplicative']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# Panel A: Kurtosis comparison
ax1.text(-0.1, 1.05, 'A', transform=ax1.transAxes, fontsize=16, fontweight='bold', va='top')

for i, (func, ftype) in enumerate(zip(funcs_ordered, types)):
 data_N1 = df_summary[(df_summary['Function'] == func) & (df_summary['N'] == 10**4)]
 data_N2 = df_summary[(df_summary['Function'] == func) & (df_summary['N'] == 10**5)]
 
 K_N1 = data_N1['K_F'].values[0]
 K_N2 = data_N2['K_F'].values[0]
 
 # Plot with offset for visibility
 x_pos = [i - 0.15, i + 0.15]
 ax1.plot(x_pos, [K_N1, K_N2], 'o-', color=colors[i], linewidth=2, markersize=8, 
 label=func, alpha=0.7)
 
 # Add values as text
 ax1.text(x_pos[0], K_N1, f'{K_N1:.1f}', ha='right', va='bottom', fontsize=8)
 ax1.text(x_pos[1], K_N2, f'{K_N2:.1f}', ha='left', va='bottom', fontsize=8)

ax1.set_xticks(range(len(funcs_ordered)))
ax1.set_xticklabels(['N=10⁴ N=10⁵'] * len(funcs_ordered))
ax1.set_ylabel('Kurtosis (K_F)', fontweight='bold')
ax1.set_title('Kurtosis: Multiplicative vs Non-Multiplicative Functions', fontweight='bold')
ax1.legend(loc='upper left', framealpha=0.9)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.axhline(y=3, color='gray', linestyle=':', alpha=0.5, label='Gaussian (K=3)')
ax1.set_ylim(0, 13)

# Add separator line
ax1.axvline(x=2.5, color='black', linestyle='--', linewidth=2, alpha=0.5)
ax1.text(1.25, 12, 'Multiplicative', ha='center', fontsize=10, fontweight='bold', 
 bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
ax1.text(3, 12, 'Non-Mult', ha='center', fontsize=10, fontweight='bold',
 bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))

# Panel B: Resonance Score comparison
ax2.text(-0.1, 1.05, 'B', transform=ax2.transAxes, fontsize=16, fontweight='bold', va='top')

for i, (func, ftype) in enumerate(zip(funcs_ordered, types)):
 data_N1 = df_summary[(df_summary['Function'] == func) & (df_summary['N'] == 10**4)]
 data_N2 = df_summary[(df_summary['Function'] == func) & (df_summary['N'] == 10**5)]
 
 RS_N1 = data_N1['RS'].values[0]
 RS_N2 = data_N2['RS'].values[0]
 
 x_pos = [i - 0.15, i + 0.15]
 ax2.plot(x_pos, [RS_N1, RS_N2], 'o-', color=colors[i], linewidth=2, markersize=8,
 label=func, alpha=0.7)
 
 ax2.text(x_pos[0], RS_N1, f'{RS_N1:.1f}', ha='right', va='bottom', fontsize=8)
 ax2.text(x_pos[1], RS_N2, f'{RS_N2:.1f}', ha='left', va='bottom', fontsize=8)

ax2.set_xticks(range(len(funcs_ordered)))
ax2.set_xticklabels(['N=10⁴ N=10⁵'] * len(funcs_ordered))
ax2.set_ylabel('Resonance Score (RS)', fontweight='bold')
ax2.set_title('Resonance Score: Multiplicative vs Non-Multiplicative Functions', fontweight='bold')
ax2.legend(loc='upper left', framealpha=0.9)
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.set_ylim(0, 7)

# Add separator line
ax2.axvline(x=2.5, color='black', linestyle='--', linewidth=2, alpha=0.5)
ax2.text(1.25, 6.5, 'Multiplicative', ha='center', fontsize=10, fontweight='bold',
 bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
ax2.text(3, 6.5, 'Non-Mult', ha='center', fontsize=10, fontweight='bold',
 bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.5))

plt.tight_layout()
plt.savefig('resonance_analysis_final.png', dpi=300, bbox_inches='tight')
print("Saved figure to: resonance_analysis_final.png")
plt.close()

print("\nFigure created successfully!")
print("=" * 80)


Saved figure to: resonance_analysis_final.png

Figure created successfully!


In [19]:

# Perform final statistical tests and prepare detailed report

print("FINAL STATISTICAL ANALYSIS")
print("=" * 80)

# Test 1: Kurtosis comparison (multiplicative vs non-multiplicative)
mult_kurtosis = df_corrected[df_corrected['type'] == 'multiplicative']['K_F'].values
non_mult_kurtosis = df_corrected[df_corrected['type'] == 'non-multiplicative']['K_F'].values

print("\n1. KURTOSIS ANALYSIS")
print("-" * 80)
print(f"Multiplicative (n={len(mult_kurtosis)}): mean={np.mean(mult_kurtosis):.4f}, "
 f"std={np.std(mult_kurtosis, ddof=1):.4f}")
print(f" Values: {mult_kurtosis}")
print(f"\nNon-multiplicative (n={len(non_mult_kurtosis)}): mean={np.mean(non_mult_kurtosis):.4f}, "
 f"std={np.std(non_mult_kurtosis, ddof=1):.4f}")
print(f" Values: {non_mult_kurtosis}")

# Mann-Whitney U test (one-sided: multiplicative > non-multiplicative)
U_kurt, p_kurt = mannwhitneyu(mult_kurtosis, non_mult_kurtosis, alternative='greater')
print(f"\nMann-Whitney U test (H1: multiplicative > non-multiplicative):")
print(f" U-statistic = {U_kurt:.2f}")
print(f" p-value = {p_kurt:.6f}")
if p_kurt < 0.05:
 print(f" *** SIGNIFICANT: Multiplicative functions have higher kurtosis (p = {p_kurt:.6f}) ***")
else:
 print(f" Not significant at α = 0.05")

# Test 2: Resonance Score comparison
mult_RS = df_corrected[df_corrected['type'] == 'multiplicative']['RS'].values
non_mult_RS = df_corrected[df_corrected['type'] == 'non-multiplicative']['RS'].values

print("\n2. RESONANCE SCORE ANALYSIS")
print("-" * 80)
print(f"Multiplicative (n={len(mult_RS)}): mean={np.mean(mult_RS):.4f}, "
 f"std={np.std(mult_RS, ddof=1):.4f}")
print(f" Values: {mult_RS}")
print(f"\nNon-multiplicative (n={len(non_mult_RS)}): mean={np.mean(non_mult_RS):.4f}, "
 f"std={np.std(non_mult_RS, ddof=1):.4f}")
print(f" Values: {non_mult_RS}")

U_RS, p_RS = mannwhitneyu(mult_RS, non_mult_RS, alternative='greater')
print(f"\nMann-Whitney U test (H1: multiplicative > non-multiplicative):")
print(f" U-statistic = {U_RS:.2f}")
print(f" p-value = {p_RS:.6f}")
if p_RS < 0.05:
 print(f" *** SIGNIFICANT: Multiplicative functions have higher RS (p = {p_RS:.6f}) ***")
else:
 print(f" Not significant at α = 0.05")

# Test 3: Check scaling behavior
print("\n3. SCALING BEHAVIOR ANALYSIS")
print("-" * 80)

for func in ['zeta', 'chi4', 'f_rand', 'L_DH (avg)']:
 data_func = df_summary[df_summary['Function'] == func]
 M_N1 = data_func[data_func['N'] == 10**4]['M_F'].values[0]
 M_N2 = data_func[data_func['N'] == 10**5]['M_F'].values[0]
 ratio = M_N2 / M_N1
 
 print(f"{func:12s}: M_F(10^5) / M_F(10^4) = {ratio:.4f}")
 
 if 'L_DH' in func:
 expected = 10**0.3085
 diff = abs(ratio - expected)
 print(f" Expected (power law): {expected:.4f}, difference: {diff:.4f}")

print("\n" + "=" * 80)
print("FINAL VERDICT")
print("=" * 80)
print("""
HYPOTHESIS: "Non-multiplicative functions (DH family) will show significantly 
higher kurtosis and resonance scores than multiplicative functions."

RESULT: HYPOTHESIS REJECTED

Evidence:
1. Kurtosis: Multiplicative > Non-multiplicative (p = {:.6f})
 - f_rand has HIGHEST kurtosis (K ≈ 11)
 - L_DH has LOWEST kurtosis (K ≈ 2.5)

2. Resonance Score: Multiplicative > Non-multiplicative (p = {:.6f})
 - f_rand has HIGHEST RS (RS ≈ 5.8)
 - L_DH has comparable RS to ζ (RS ≈ 2.4)

3. No power-law peak growth observed for DH:
 - Expected ratio: 2.03 (for σ = 0.8085)
 - Observed ratio: 1.00

INTERPRETATION:
The random multiplicative function shows the STRONGEST heavy-tailed behavior,
not the non-multiplicative DH function. This contradicts the core premise that
non-multiplicativity leads to heavier-tailed distributions and higher resonance.

The DH function's known off-line zeros do NOT manifest as expected in the 
D_F(t; N) partial sum statistics over the sampled height range [80, 190].
""".format(p_kurt, p_RS))
print("=" * 80)


FINAL STATISTICAL ANALYSIS

1. KURTOSIS ANALYSIS
--------------------------------------------------------------------------------
Multiplicative (n=6): mean=5.9101, std=3.7858
 Values: [ 2.56548176 5.16932354 9.44306179 1.80044658 5.10353919 11.37874796]

Non-multiplicative (n=6): mean=2.4926, std=0.0085
 Values: [2.50032327 2.50049733 2.50009128 2.48487327 2.48506088 2.48462502]

Mann-Whitney U test (H1: multiplicative > non-multiplicative):
 U-statistic = 30.00
 p-value = 0.032468
 *** SIGNIFICANT: Multiplicative functions have higher kurtosis (p = 0.032468) ***

2. RESONANCE SCORE ANALYSIS
--------------------------------------------------------------------------------
Multiplicative (n=6): mean=3.8150, std=1.4152
 Values: [2.41739807 3.91252107 4.88328665 2.09378351 3.78104538 5.80166886]

Non-multiplicative (n=6): mean=2.4224, std=0.0036
 Values: [2.41913612 2.41907371 2.41922868 2.42564434 2.42573653 2.4255232 ]

Mann-Whitney U test (H1: multiplicative > non-multiplicative):
 U-s

In [20]:

# Fix JSON serialization issue
metadata = {
 "analysis_type": "Resonance detection in L-functions",
 "objective": "Test if non-multiplicative functions show higher kurtosis and resonance scores",
 "hypothesis": "Non-multiplicative (DH) >> Multiplicative (zeta, chi4, f_rand)",
 "result": "REJECTED - Multiplicative (especially f_rand) >> Non-multiplicative",
 "functions_analyzed": {
 "multiplicative": ["zeta", "chi4", "f_rand"],
 "non_multiplicative": ["L_DH", "L_DH_pert1", "L_DH_pert2"]
 },
 "parameters": {
 "N_values": [10000, 100000],
 "t_range": [80, 190],
 "delta_t_formula": "2*pi / log(N)",
 "summation_algorithm": "Kahan compensated summation"
 },
 "known_DH_zeros": [
 {"sigma": 0.8085, "t": 85.70},
 {"sigma": 0.6508, "t": 114.16},
 {"sigma": 0.5744, "t": 166.48},
 {"sigma": 0.7243, "t": 176.70}
 ],
 "metrics_computed": [
 "M_F (maximum magnitude)",
 "V_F (mean square)",
 "K_F (kurtosis)",
 "RS (resonance score)",
 "P_2, P_3, P_4, P_5 (tail probabilities)"
 ],
 "statistical_tests": {
 "test_used": "Mann-Whitney U test (non-parametric)",
 "significance_level": 0.05,
 "kurtosis_comparison": {
 "p_value": float(p_kurt),
 "significant": bool(p_kurt < 0.05),
 "direction": "multiplicative > non-multiplicative"
 },
 "resonance_score_comparison": {
 "p_value": float(p_RS),
 "significant": bool(p_RS < 0.05)
 }
 },
 "key_findings": {
 "kurtosis_highest": "f_rand (K ≈ 10-11)",
 "kurtosis_lowest": "L_DH (K ≈ 2.5)",
 "RS_highest": "f_rand (RS ≈ 5.8)",
 "RS_lowest": "L_DH (RS ≈ 2.4)",
 "DH_scaling_expected": 2.03,
 "DH_scaling_observed": 1.00
 }
}

import json
with open('analysis_metadata.json', 'w') as f:
 json.dump(metadata, f, indent=2)

print("Saved metadata to: analysis_metadata.json")
print("\nAll analysis files saved:")
print(" 1. resonance_metrics_summary.csv")
print(" 2. resonance_metrics_full.csv")
print(" 3. resonance_analysis_final.png")
print(" 4. analysis_metadata.json")


Saved metadata to: analysis_metadata.json

All analysis files saved:
 1. resonance_metrics_summary.csv
 2. resonance_metrics_full.csv
 3. resonance_analysis_final.png
 4. analysis_metadata.json


In [21]:

# Create final summary report with discretionary decisions

print("=" * 80)
print("DISCRETIONARY ANALYTICAL DECISIONS")
print("=" * 80)
print("""
1. Truncation lengths N:
 Selected N ∈ {10^4, 10^5} rather than including 10^6 due to computational
 constraints and time limitations. The full research program specifies up to
 10^6, but the pattern is already clear at 10^5.

2. Height range t:
 Selected t ∈ [80, 190] to cover known DH zeros at t ≈ 85.7, 114.2, 166.5, 176.7.
 This is narrower than the full range specified in the research program but
 captures the critical region near known off-line zeros.

3. Sampling resolution:
 Used Δt = 2π/log(N) as specified in the research program, giving:
 - For N = 10^4: Δt ≈ 0.68
 - For N = 10^5: Δt ≈ 0.55
 This follows the Nyquist-like resolution recommendation.

4. Statistical test selection:
 Used Mann-Whitney U test (non-parametric) rather than t-test because:
 - No assumption of normality required
 - Robust to outliers and heavy-tailed distributions
 - More appropriate for small sample sizes (n=6 per group)

5. Significance threshold:
 Set α = 0.05 (standard convention in scientific research)

6. DH perturbations:
 Used ε ∈ {-0.01, 0, +0.01} for perturbed DH functions to test robustness
 beyond the single DH counterexample.

7. Random seed:
 Used seed=42 for reproducibility of random multiplicative function f_rand.
 This is arbitrary but fixed for reproducibility.

8. Kahan summation:
 Implemented Kahan compensated summation algorithm to minimize floating-point
 error accumulation in the D_F computation. This is critical for numerical
 stability with N ~ 10^5 terms.

9. Effect size calculation:
 Used Cohen's d for effect size estimation with pooled standard deviation,
 which is standard for comparing two independent groups.

10. Figure design:
 Created two-panel vertical layout (stacked subfigures) to show both kurtosis
 and resonance score comparisons, following the instruction for 2 subfigures
 in single-column layout.
""")
print("=" * 80)


DISCRETIONARY ANALYTICAL DECISIONS

1. Truncation lengths N:
 Selected N ∈ {10^4, 10^5} rather than including 10^6 due to computational
 constraints and time limitations. The full research program specifies up to
 10^6, but the pattern is already clear at 10^5.

2. Height range t:
 Selected t ∈ [80, 190] to cover known DH zeros at t ≈ 85.7, 114.2, 166.5, 176.7.
 This is narrower than the full range specified in the research program but
 captures the critical region near known off-line zeros.

3. Sampling resolution:
 Used Δt = 2π/log(N) as specified in the research program, giving:
 - For N = 10^4: Δt ≈ 0.68
 - For N = 10^5: Δt ≈ 0.55
 This follows the Nyquist-like resolution recommendation.

4. Statistical test selection:
 Used Mann-Whitney U test (non-parametric) rather than t-test because:
 - No assumption of normality required
 - Robust to outliers and heavy-tailed distributions
 - More appropriate for small sample sizes (n=6 per group)

5. Significance threshold:
 Set α = 0.05 (st